### DECS<->VISA example

Here a demonstration of working with DECS<->VISA from a jupyter notebook is provided.

The general operation is very similar to the `example_client.py` code, but rather than require a terminal be opened to launch `decs_visa.py` we will do it directly from the notebook using a python subprocess.

*Note: When running on Windows, outputs from DECS<->VISA are stored in a `decs_visa.log` file, which will be created in your working directory. If struggling to establish a connection between DECS<->VISA and oi.DECS, check the information captured in the `decs_visa.log` file.*

In [1]:
import time
import subprocess
import platform
import Proteox_helpers as ph

decs_visa_path = './decs_visa.py'

# Start the decs_visa subprocess
running_on = platform.platform()
if running_on.startswith("Windows"):
    print(f"Running on {running_on} - start subprocess without PIPEd output")
    subprocess.Popen(["python", decs_visa_path])
else:
    print(f"Running on {running_on} - start subprocess with PIPEd output")
    subprocess.Popen(["python3", decs_visa_path], stdout=subprocess.PIPE)
    
# short delay to allow for the server to open
time.sleep(1)

Running on Windows-10-10.0.26200-SP0 - start subprocess without PIPEd output


Once the server is open we can connect to it in the same way as the `client_example.py` code - as that has already worked, we'll skip some of the error checking here for brevity.

In [2]:
import pyvisa as visa

rm = visa.ResourceManager('@py')
decs_visa_server_ip = "localhost"
decs_visa_server_port = "33576"
pyvisa_connection = f"TCPIP0::{decs_visa_server_ip}::{decs_visa_server_port}::SOCKET"
decs_visa = rm.open_resource(pyvisa_connection)
decs_visa.read_termination = "\n"
decs_visa.write_termination = "\n"
decs_visa.chunk_size = 204800
decs_visa.timeout = 10000


try:
    print(decs_visa.query("*IDN?"))
except Exception as e:
    print(e)

QD - Oxford, DECS, decs-55fff5, 1.7.2.9015


In [ ]:
# Define some test queries (get_ and set_) to send to
# the socket server
# get_test ="get_MC_T"
get_tank_p = "get_P1_P"
get_still_p = "get_P2_P"
get_foreline_p = "get_P3_P"
tank_p_mbar = 0
still_p_mbar = 0
foreline_p_mbar = 0
fp_p_mbar = 0
cp_p_mbar = 0

# get_NT_AV05 = "get_AV05"
# set_NT_AV05 = "set_AV05"
# pulse_valve = "get_pulse_AV05"
on = 1.0
off = 0.0

In [4]:
decs_visa.write("SHUTDOWN")

9

In [ ]:
try:
    tank_p_mbar = float(decs_visa.query(get_tank_p))/100
    still_p_mbar = float(decs_visa.query(get_still_p))/100
    foreline_p_mbar = float(decs_visa.query(get_foreline_p))/100
        
    print(f"Tank Pressure = {tank_p_mbar} mBar")
    print(f"Still Pressure => {still_p_mbar} mBar")
    print(f"Foreline Pressure => {foreline_p_mbar} mBar")

    time.sleep(5)
except (visa.errors.VisaIOError, ConnectionResetError) as e:
    print(e)
    print("Communication issue with DECS<->VISA server")
    sys.exit(1)

In [ ]:
# get temperature
decs_visa.query("get_MC_T")

In [ ]:
print("CP state")
decs_visa.query("get_CP")

In [ ]:
print("CP on")
decs_visa.query("set_CP:1.0")
time.sleep(2)
decs_visa.query("set_FP:1.0")

In [ ]:
decs_visa.query("set_FP:0.0")
print("CP off")
decs_visa.query("set_CP:0.0")

In [ ]:
print("AV03 state")
decs_visa.query("set_AV03:1")

In [3]:
# get Proteox record
decs_visa.query("get_PRXS")

'0'

In [ ]:
# example with valve actuation
# updated both command_parser.py and command_dictionary.py to include the valve states

try:
    print("Closed AV05")
    decs_visa.query("set_AV05:0.0") #works
    # decs_visa.query(f"{set_NT_AV05}:0.0") #works
    time.sleep(2)
    # decs_visa.query(f"set_valve:1.0")
    decs_visa.query("set_AV05:1.0")
    # decs_visa.query(f"{set_NT_AV05}:1.0") #works
    print("Opened AV05")
    # print("Pulsed AV05")
    # decs_visa.query(f"pulse_valve")
except (visa.errors.VisaIOError, ConnectionResetError) as e:
    print(e)
    print("Communication issue with DECS<->VISA server")
    sys.exit(1)

In [ ]:
# example with pump control
# updated both command_parser.py and command_dictionary.py to include the pump states

try:
    fp_p_mbar = float(decs_visa.query("get_P4_P"))/100
    cp_p_mbar = float(decs_visa.query("get_P5_P"))/100
        
    print(f"Forepump Pressure = {fp_p_mbar} mBar")
    print(f"Compressor Pressure => {cp_p_mbar} mBar")

    print("Turn on FP")
    decs_visa.query("set_FP:1.0")
    time.sleep(2)
    fp_p_mbar = float(decs_visa.query("get_P4_P"))/100
    cp_p_mbar = float(decs_visa.query("get_P5_P"))/100
        
    print(f"Forepump Pressure = {fp_p_mbar} mBar")
    print(f"Compressor Pressure => {cp_p_mbar} mBar")
    time.sleep(2)
    decs_visa.query("set_FP:0.0")
    print("Turn off FP")
except (visa.errors.VisaIOError, ConnectionResetError) as e:
    print(e)
    print("Communication issue with DECS<->VISA server")
    sys.exit(1)

In [ ]:
# change still heater setpoint
try:
    ph.set_still_heater_power(decs_visa, 10e-3)  # example: 100 uW

    for i in range(60):
        T_mK = float(decs_visa.query("get_MC_T")) * 1000
        T_still_mK = float(decs_visa.query("get_STILL_T")) * 1000
        # flow = float(decs_visa.query("get_3He_F"))
        print(f"MC T = {T_mK:.2f} mK")
        print(f"Still T = {T_still_mK:.2f} mK")
        # print(f"He3 Flow = {flow:.2f}")
        time.sleep(60)

finally:
    ph.all_heaters_off(decs_visa)

In [ ]:
# change MC Temperature setpoint
# change still heater setpoint
try:
    ph.set_mc_temperature(decs_visa, 50e-3)  # example: 100 mK
    # ph.set_still_heater_power(decs_visa, 10e-3)  # example: 100 uW

    for i in range(10):
        T_mK = float(decs_visa.query("get_MC_T")) * 1000
        T_still_mK = float(decs_visa.query("get_STILL_T")) * 1000
        # flow = float(decs_visa.query("get_3He_F"))
        print(f"MC T = {T_mK:.2f} mK")
        print(f"Still T = {T_still_mK:.2f} mK")
        # print(f"He3 Flow = {flow:.2f}")
        time.sleep(30)

finally:
    ph.all_heaters_off(decs_visa)

In [ ]:
# set a range of MC Temperature setpoint
# change still heater setpoint
repeat_n = 10
mc_setpoint = 5E-3
try:
    ph.set_mc_temperature(decs_visa, mc_setpoint)  # example: 100 mK
    # ph.set_still_heater_power(decs_visa, 10e-3)  # example: 100 uW
    
    for i in range(repeat_n):
        T_mK = float(decs_visa.query("get_MC_T")) * 1000
        T_still_mK = float(decs_visa.query("get_STILL_T")) * 1000
        T_sp_mK = mc_setpoint * 1000
        mc_setpoint = mc_setpoint + (i*1E-3)
        ph.set_mc_temperature(decs_visa, mc_setpoint)
        # flow = float(decs_visa.query("get_3He_F"))
        print(f"Change MC temperature to: {T_sp_mK} mK")
        print(f"MC T = {T_mK:.2f} mK")
        print(f"Still T = {T_still_mK:.2f} mK")
        # print(f"He3 Flow = {flow:.2f}")
        time.sleep(120)

finally:
    ph.all_heaters_off(decs_visa)

In [ ]:
# set temperature
decs_visa.query("set_MC_T:0.050")

In [ ]:
# get still temperature
decs_visa.query("get_STILL_T")

In [ ]:
# set still heater
decs_visa.query("set_STILL_H:0.002")

In [ ]:
# set MC heater
decs_visa.query("set_MC_H:0.000012")

In [ ]:
# set heaters off
decs_visa.query("set_MC_H_OFF:0")
decs_visa.query("set_STILL_H_OFF:0")

**NB** If the query above threw and exception you'll need to check the server settings, and the DECS<->VISA logs to determine why...

Assuming the above worked correctly; run a simple loop to pull back some data from the system:

In [ ]:
import matplotlib.pyplot as plt

pt2_temps = []
time_stamps = []
for i in range(20):
    time_stamps.append(time.time())
    pt2_temps.append(float(decs_visa.query("get_PT2_T1").strip()))
    time.sleep(1)

fig, ax = plt.subplots()
fig.set_facecolor("darkgrey")
ax.set_facecolor('black')
plt.xlabel(r'Elapsed time [s]')
plt.ylabel(r'PT2 temperature [K]')
plt.grid(visible=True, which = 'major', color = 'grey', linestyle = '--')
plt.scatter(time_stamps, pt2_temps)
plt.show()

We'll want to stop the DECS<->VISA process once we're done, so

In [ ]:
decs_visa.write("SHUTDOWN")

In [ ]:
# set PRXS to Bypass off circulation
# TP speed will start to slow down when flow is high where still pressure rises above 2
# Check status of CP and bypass valve
